# Step 1: Parsing File `.lin` → CSV

Memproses file `.lin` dari BBO dan mengekstrak:
- **Hand 4 pemain** (North, South, East, West)
- **Vulnerability** dan **dealer**
- **Bidding sequence** (urutan bid seluruh pemain)
- **Kontrak final** dari bidding

Output: `data/parsed/parsed_boards.csv`

**Prasyarat:** File `.lin` tersedia di `data/raw/`

In [8]:
import sys
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt

DATA_RAW       = Path(ROOT) / "data" / "raw"
DATA_PARSED    = Path(ROOT) / "data" / "parsed"
DATA_PROCESSED = Path(ROOT) / "data" / "processed"
RESULTS        = Path(ROOT) / "results"

DATA_PARSED.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
(RESULTS / "figures").mkdir(parents=True, exist_ok=True)
(RESULTS / "metrics").mkdir(parents=True, exist_ok=True)

PARSED_CSV = DATA_PARSED / "parsed_boards.csv"

print(f"Root proyek : {ROOT}")
print(f"File .lin   : {len(list(DATA_RAW.glob('*.lin')))} file")
print("Setup selesai.")

Root proyek : d:\SkripsiBBO
File .lin   : 411 file
Setup selesai.


## 1.1 Eksplorasi File `.lin`

Lihat file yang tersedia sebelum diproses.

In [9]:
lin_files = sorted(DATA_RAW.glob("*.lin"))
print(f"Total file .lin: {len(lin_files)}")
print("\n5 file pertama:")
for f in lin_files[:5]:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<25} ({size_kb:.1f} KB)")

Total file .lin: 411

5 file pertama:
  85168.lin                 (9.9 KB)
  85169.lin                 (6.1 KB)
  85170.lin                 (6.1 KB)
  85171.lin                 (9.0 KB)
  85173.lin                 (7.9 KB)


In [10]:
# Lihat konten mentah satu file .lin (token pertama)
if lin_files:
    sample_lin = lin_files[0]
    print(f"Contoh isi {sample_lin.name}:")
    print("-" * 60)
    lines = sample_lin.read_text(encoding="utf-8", errors="ignore").split("|")
    preview = "|".join(lines[:30]) + "|"
    print(preview)
else:
    print("Tidak ada file .lin ditemukan di data/raw/")

Contoh isi 85168.lin:
------------------------------------------------------------
vg|NORWEGIAN 1_ DIVISION,10:2_11,I,1,24,BOGØ,21,THORESEN,21|
rs|,,,,,,,,,,,,,,,,,,,,,,,,3NW-1,3NW-1,5CEx=,5CEx=,3NE-1,3NE-1,3NE=,3NE=,3NE+1,3NE=,6HN=,4HS+2,6SW=,6SW-1,1NS+3,1NS+4,3NS-1,3NN+1,2DS+1,2NS=,2HN+1,2HN+1,3NS+1,3NS+1|
pn|Helgemo,Thoresen,Bogø,E. Eide,Høyland,Kristoffer,Ovesen,Karlberg|pg||
qx|o13|st||md|3SJ9HAQ9DQJT96CJ95,SA53HK84DAK54CKT8,SKT87HT76D873C743,SQ642HJ532D2CAQ62|sv|b|mb|p|mb|p|mb|1D|mb|1N|mb|p|mb|2C|mb|p|


## 1.2 Parsing Semua File `.lin`

Menjalankan parser untuk mengekstrak semua board.

In [11]:
from parser import collect_rows, write_csv

print("Memproses file .lin...")
rows = collect_rows(input_dir=DATA_RAW, limit=None)
write_csv(rows=rows, output_csv=PARSED_CSV)
print(f"\nHasil: {len(rows)} board record dari {len(lin_files)} file")
print(f"Disimpan ke: {PARSED_CSV}")

Memproses file .lin...

Hasil: 8632 board record dari 411 file
Disimpan ke: d:\SkripsiBBO\data\parsed\parsed_boards.csv


## 1.3 Validasi Hasil Parsing

In [12]:
df_parsed = pd.read_csv(PARSED_CSV)
print(f"Shape: {df_parsed.shape}")
print(f"\nKolom: {list(df_parsed.columns)}")
df_parsed.head(3)

Shape: (8632, 19)

Kolom: ['source_file', 'source_type', 'room', 'board_number', 'vulnerability_code', 'vulnerability', 'dealer_code', 'dealer', 'south_hand_raw', 'west_hand_raw', 'north_hand_raw', 'east_hand_raw', 'south_hand_norm', 'west_hand_norm', 'north_hand_norm', 'east_hand_norm', 'tricks_claimed', 'bidding_sequence', 'final_contract']


,source_file,source_type,room,board_number,vulnerability_code,vulnerability,dealer_code,dealer,south_hand_raw,west_hand_raw,north_hand_raw,east_hand_raw,south_hand_norm,west_hand_norm,north_hand_norm,east_hand_norm,tricks_claimed,bidding_sequence,final_contract
0,85168.lin,BBO,open,13,b,both,3.0,N,SJ9HAQ9DQJT96CJ95,SA53HK84DAK54CKT8,SKT87HT76D873C743,SQ642HJ532D2CAQ62,SJ9HAQ9DQJT96CJ95,SA53HK84DAK54CKT8,SKT87HT76D873C743,SQ642HJ532D2CAQ62,8.0,PASS PASS 1D 1NT PASS 2C PASS 2D! PASS 2NT PAS...,3NT
1,85168.lin,BBO,closed,13,b,both,3.0,N,SJ9HAQ9DQJT96CJ95,SA53HK84DAK54CKT8,SKT87HT76D873C743,SQ642HJ532D2CAQ62,SJ9HAQ9DQJT96CJ95,SA53HK84DAK54CKT8,SKT87HT76D873C743,SQ642HJ532D2CAQ62,NaN,PASS PASS 1D 1NT PASS 2C! PASS 2D PASS 3NT PAS...,3NT
2,85168.lin,BBO,open,14,o,none,4.0,E,SAT542HAJ6DA983C4,SHQ98743DK2CJ9863,SQJ983HKT2DT75CQT,SK76H5DQJ64CAK752,SAT542HAJ6DA983C4,SHQ98743DK2CJ9863,SQJ983HKT2DT75CQT,SK76H5DQJ64CAK752,11.0,1C 1S X 2NT PASS 4S 5C PASS PASS X PASS PASS PASS,5C


In [13]:
print("=== Validasi Parsing ===")
print(f"Total board   : {len(df_parsed)}")
print(f"Missing dealer: {df_parsed['dealer'].isna().sum()}")
print(f"Missing bidding: {(df_parsed['bidding_sequence'] == '').sum()}")
print(f"Pass-out boards: {(df_parsed['final_contract'] == 'PASSOUT').sum()}")
print(f"\nVulnerability distribution:")
print(df_parsed["vulnerability"].value_counts())
print(f"\nDealer distribution:")
print(df_parsed["dealer"].value_counts())

=== Validasi Parsing ===
Total board   : 8632
Missing dealer: 7
Missing bidding: 0
Pass-out boards: 90

Vulnerability distribution:
vulnerability
both    2222
ew      2167
none    2135
ns      2108
Name: count, dtype: int64

Dealer distribution:
dealer
E    2220
N    2215
S    2109
W    2081
Name: count, dtype: int64


In [14]:
sample = df_parsed.iloc[0]
print("=== Contoh Board ===")
print(f"File      : {sample['source_file']}")
print(f"Board     : {sample['board_number']} ({sample['room']})")
print(f"Dealer    : {sample['dealer']}")
print(f"Vuln      : {sample['vulnerability']}")
print(f"North     : {sample['north_hand_norm']}")
print(f"South     : {sample['south_hand_norm']}")
print(f"East      : {sample['east_hand_norm']}")
print(f"West      : {sample['west_hand_norm']}")
print(f"Bidding   : {sample['bidding_sequence']}")
print(f"Kontrak   : {sample['final_contract']}")

=== Contoh Board ===
File      : 85168.lin
Board     : 13 (open)
Dealer    : N
Vuln      : both
North     : SKT87HT76D873C743
South     : SJ9HAQ9DQJT96CJ95
East      : SQ642HJ532D2CAQ62
West      : SA53HK84DAK54CKT8
Bidding   : PASS PASS 1D 1NT PASS 2C PASS 2D! PASS 2NT PASS 3NT PASS PASS PASS
Kontrak   : 3NT


---
## Output

File yang dihasilkan:
- `data/parsed/parsed_boards.csv` — hasil parsing semua file `.lin`

**Langkah berikutnya:** Buka `02_eda.ipynb` untuk eksplorasi data.